# v3.0.0 PD AST Spectrograms — Primary Experiment

Primary PD screening experiment on Bridge2AI v3.0.0 (833 participants).
Adapted from v2.0.1 pipeline with key changes:
- Path: `features/torchaudio_mel_spectrogram.parquet` (column: `mel_spectrogram`)
- Mel bins: 60 (resized to 128 for AST)
- Hierarchical phenotype: `phenotype/diagnosis/parkinsons_disease.tsv` + `control.tsv`
- Task selection: 8 tasks with adequate PD + Control coverage
- Participant IDs: 6-digit numeric (zero-padded)

In [ ]:
#1 - Imports & paths
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

ROOT = Path('/data0/b2ai-voice/3.0.0')
SPEC = ROOT / 'features' / 'torchaudio_mel_spectrogram.parquet'
PD_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'parkinsons_disease.tsv'
CTRL_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'control.tsv'
DEMO_PATH = ROOT / 'phenotype' / 'demographics' / 'demographics.tsv'
RESULTS_DIR = Path('/home/saptpurk/bridge2ai-voice-parkinsons-ast/results/v3')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Spec exists: {SPEC.exists()}')
print(f'PD phen exists: {PD_PHEN.exists()}')
print(f'Ctrl phen exists: {CTRL_PHEN.exists()}')

In [ ]:
#2 - Load spectrogram data (row-group reads for memory efficiency)
pf = pq.ParquetFile(SPEC)
parts = []
for i in range(pf.num_row_groups):
    parts.append(pf.read_row_group(i, columns=['participant_id','session_id','task_name','mel_spectrogram','n_frames']).to_pandas())
spec = pd.concat(parts, ignore_index=True)

# Zero-pad participant IDs to 6 digits
spec['participant_id'] = spec['participant_id'].astype(str).str.zfill(6)

print(f'Total recordings: {len(spec)}')
print(f'Unique participants: {spec["participant_id"].nunique()}')
print(f'Unique tasks: {spec["task_name"].nunique()}')

In [ ]:
#3 - Build PD labels from hierarchical phenotype
pd_df = pd.read_csv(PD_PHEN, sep='\t')
ctrl_df = pd.read_csv(CTRL_PHEN, sep='\t')

pd_ids = set(pd_df['participant_id'].astype(str).str.zfill(6))
ctrl_ids = set(ctrl_df['participant_id'].astype(str).str.zfill(6))

# Remove any PD-Control overlap
overlap = pd_ids & ctrl_ids
ctrl_ids_clean = ctrl_ids - overlap
print(f'PD participants: {len(pd_ids)}')
print(f'Control participants (clean): {len(ctrl_ids_clean)}')
print(f'Overlap removed: {len(overlap)}')

# Assign labels
spec['label'] = np.nan
spec.loc[spec['participant_id'].isin(pd_ids), 'label'] = 1
spec.loc[spec['participant_id'].isin(ctrl_ids_clean), 'label'] = 0

# Keep only labeled recordings
data = spec.dropna(subset=['label']).copy()
data['label'] = data['label'].astype(int)
print(f'\nLabeled recordings: {len(data)}')
print(f'PD recordings: {(data["label"]==1).sum()}')
print(f'Control recordings: {(data["label"]==0).sum()}')
print(f'Unique participants: {data["participant_id"].nunique()}')

In [ ]:
#4 - Task selection: tasks with adequate PD + Control coverage
# Selected based on data exploration: tasks where both PD and Control
# participants have substantial representation
SELECTED_TASKS = [
    'prolonged-vowel',           # 105 PD, 147 Ctrl - sustained phonation
    'glides-high-to-low',        # 104 PD, 147 Ctrl - pitch control
    'glides-low-to-high',        # 104 PD, 147 Ctrl - pitch control
    'diadochokinesis-pataka',    # 94 PD, 125 Ctrl - articulatory agility
    'rainbow-passage',           # 94 PD, 125 Ctrl - connected speech
    'picture-description',       # 93 PD, 123 Ctrl - cognitive-linguistic
    'story-recall',              # 93 PD, 123 Ctrl - memory + speech
    'maximum-phonation-time-1',  # 95 PD, 125 Ctrl - sustained phonation
]

MIN_TIME_FRAMES = 100

data_selected = data[
    (data['task_name'].isin(SELECTED_TASKS)) &
    (data['n_frames'] >= MIN_TIME_FRAMES)
].copy()

print(f'Selected task recordings: {len(data_selected)}')
print(f'PD: {(data_selected["label"]==1).sum()} recordings from {data_selected[data_selected["label"]==1]["participant_id"].nunique()} participants')
print(f'Ctrl: {(data_selected["label"]==0).sum()} recordings from {data_selected[data_selected["label"]==0]["participant_id"].nunique()} participants')
print(f'Total participants: {data_selected["participant_id"].nunique()}')
print(f'\nPer-task counts:')
print(data_selected.groupby('task_name')['label'].value_counts().unstack(fill_value=0))

In [ ]:
#5 - Process spectrograms
from tqdm import tqdm

TARGET_SEQ_LEN = 1024

def process_spectrogram(spec_raw, target_len=1024):
    """Process raw spectrogram with reflect padding/center crop."""
    spec = np.stack(spec_raw).astype(np.float32)
    n_mels, time_len = spec.shape
    if time_len < target_len:
        pad_width = target_len - time_len
        spec = np.pad(spec, ((0, 0), (0, pad_width)), mode='reflect')
    elif time_len > target_len:
        start = (time_len - target_len) // 2
        spec = spec[:, start:start + target_len]
    return spec

X_list = []
for idx, row in tqdm(data_selected.iterrows(), total=len(data_selected), desc='Processing'):
    processed = process_spectrogram(row['mel_spectrogram'], TARGET_SEQ_LEN)
    X_list.append(processed)

X_raw = np.stack(X_list)
y_raw = data_selected['label'].values
participants_raw = data_selected['participant_id'].values
tasks_raw = data_selected['task_name'].values

print(f'Processed shape: {X_raw.shape}')  # (N, 60, 1024)
print(f'Value range: [{X_raw.min():.2f}, {X_raw.max():.2f}]')

In [ ]:
#6 - AST model definition and helpers
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import ASTModel, ASTConfig
from scipy.ndimage import zoom

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def resize_spectrogram(spec, target_mel=128, target_time=1024):
    """Resize spectrogram to AST expected dimensions (60 -> 128 mel bins)."""
    current_mel, current_time = spec.shape
    mel_ratio = target_mel / current_mel
    time_ratio = target_time / current_time
    resized = zoom(spec, (mel_ratio, time_ratio), order=1)
    return resized.astype(np.float32)

class ASTClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, freeze_base=False):
        super().__init__()
        if pretrained:
            self.ast = ASTModel.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
            hidden_size = self.ast.config.hidden_size
        else:
            config = ASTConfig(hidden_size=768, num_hidden_layers=12,
                             num_attention_heads=12, intermediate_size=3072,
                             max_length=1024, num_mel_bins=128)
            self.ast = ASTModel(config)
            hidden_size = 768
        if freeze_base:
            for param in self.ast.parameters():
                param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = x.transpose(1, 2)  # (B, 128, 1024) -> (B, 1024, 128)
        outputs = self.ast(input_values=x)
        pooled = outputs.pooler_output
        return self.classifier(pooled)

class ASTDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, participants, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.participants = np.array(participants)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            if np.random.random() < 0.5:
                t = np.random.randint(50, 150)
                t0 = np.random.randint(0, max(1, x.shape[1] - t))
                x[:, t0:t0+t] = 0
            if np.random.random() < 0.5:
                f = np.random.randint(10, 30)
                f0 = np.random.randint(0, max(1, x.shape[0] - f))
                x[f0:f0+f, :] = 0
        return {'inputs': x, 'labels': self.y[idx], 'participant': self.participants[idx]}

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

def evaluate_fold(model, loader, device):
    model.eval()
    all_probs, all_labels, all_parts = [], [], []
    with torch.no_grad():
        for batch in loader:
            inputs = batch['inputs'].to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch['labels'].numpy())
            all_parts.extend(batch['participant'])
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_parts = np.array(all_parts)
    unique_parts = np.unique(all_parts)
    part_probs, part_labels = [], []
    for p in unique_parts:
        mask = all_parts == p
        part_probs.append(all_probs[mask].mean())
        part_labels.append(all_labels[mask][0])
    return np.array(part_probs), np.array(part_labels), unique_parts

print('Model and helpers defined.')

In [ ]:
#7 - 5-Fold Cross-Validation (idempotent: load cached checkpoints when available)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve
import copy
import time

unique_participants = np.unique(participants_raw)
participant_labels = np.array([y_raw[participants_raw == p][0] for p in unique_participants])

print(f'Total participants: {len(unique_participants)}')
print(f'PD: {int(participant_labels.sum())} ({participant_labels.mean():.1%})')
print(f'Control: {int((participant_labels == 0).sum())}')

N_FOLDS = 5

# Idempotency: if all 5 fold checkpoints + cv_results.npz exist, restore in-memory state and skip training.
_cv_npz = RESULTS_DIR / 'ast_pd_v3_cv_results.npz'
_ckpts = [RESULTS_DIR / f'ast_pd_v3_fold{k}.pt' for k in range(1, N_FOLDS + 1)]
_have_all = _cv_npz.exists() and all(p.exists() for p in _ckpts)

if _have_all:
    print(f'\n[CACHED] All {N_FOLDS} fold checkpoints + CV results exist - loading and skipping training.')
    _cached = np.load(_cv_npz, allow_pickle=True)
    saved_pids = _cached['participant_ids']
    _pid_to_idx = {p: i for i, p in enumerate(saved_pids)}
    _order = np.array([_pid_to_idx[p] for p in unique_participants])
    all_oof_probs = _cached['oof_probs'][_order].astype(np.float32)
    all_oof_labels = _cached['oof_labels'][_order].astype(np.int64)
    fold_results = []
    for i in range(N_FOLDS):
        fold_results.append({
            'fold': i + 1,
            'auc': float(_cached['fold_aucs'][i]),
            'f1_opt': float(_cached['fold_f1s'][i]),
            'recall_opt': float(_cached['fold_recalls'][i]),
            'precision_opt': float(_cached['fold_precisions'][i]),
            'threshold': float(_cached['fold_thresholds'][i]),
        })
    norm_stats = {i + 1: {'mean': float(_cached['norm_means'][i]), 'std': float(_cached['norm_stds'][i])} for i in range(N_FOLDS)}
    print(f'Restored: all_oof_probs ({len(all_oof_probs)}), {len(fold_results)} fold results, {len(norm_stats)} fold norm stats.')
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    fold_results = []
    all_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
    all_oof_labels = participant_labels.astype(np.int64).copy()
    norm_stats = {}

    total_start = time.time()

    for fold, (train_idx, val_idx) in enumerate(skf.split(unique_participants, participant_labels)):
        print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')

        train_parts_fold = unique_participants[train_idx]
        val_parts_fold = unique_participants[val_idx]

        train_mask = np.isin(participants_raw, train_parts_fold)
        val_mask = np.isin(participants_raw, val_parts_fold)

        X_train_fold = X_raw[train_mask]
        y_train_fold = y_raw[train_mask]
        parts_train_fold = participants_raw[train_mask]

        X_val_fold = X_raw[val_mask]
        y_val_fold = y_raw[val_mask]
        parts_val_fold = participants_raw[val_mask]

        print(f'Train: {len(X_train_fold)} recordings from {len(train_parts_fold)} participants')
        print(f'Val: {len(X_val_fold)} recordings from {len(val_parts_fold)} participants')

        X_train_ast = np.stack([resize_spectrogram(x) for x in tqdm(X_train_fold, desc='resize train', leave=False)])
        X_val_ast = np.stack([resize_spectrogram(x) for x in tqdm(X_val_fold, desc='resize val', leave=False)])

        fold_mean = X_train_ast.mean()
        fold_std = X_train_ast.std()
        X_train_ast = (X_train_ast - fold_mean) / (fold_std + 1e-8)
        X_val_ast = (X_val_ast - fold_mean) / (fold_std + 1e-8)
        norm_stats[fold + 1] = {'mean': float(fold_mean), 'std': float(fold_std)}

        train_ds = ASTDataset(X_train_ast, y_train_fold, parts_train_fold, augment=True)
        val_ds = ASTDataset(X_val_ast, y_val_fold, parts_val_fold, augment=False)

        class_counts = np.bincount(y_train_fold)
        weights = 1.0 / class_counts
        sample_weights = weights[y_train_fold]
        sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))

        train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=4, pin_memory=True)
        val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

        model = ASTClassifier(num_classes=2, pretrained=True, freeze_base=False).to(device)

        backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
        head_params = [p for n, p in model.named_parameters() if 'classifier' in n]
        optimizer = torch.optim.AdamW([
            {'params': backbone_params, 'lr': 5e-6, 'weight_decay': 0.01},
            {'params': head_params, 'lr': 5e-4, 'weight_decay': 0.01}
        ], betas=(0.9, 0.999))
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-7)

        cc = np.bincount(y_train_fold)
        cw = (cc.sum() / (2.0 * cc)).astype(np.float32)
        class_weights_fold = torch.tensor(cw, dtype=torch.float32).to(device)
        criterion = FocalLoss(alpha=class_weights_fold, gamma=2.0)
        print(f'Class weights: {cw}')

        best_score = 0
        best_state = None
        patience_counter = 0
        PATIENCE = 10

        for epoch in range(30):
            model.train()
            total_loss = 0
            for batch in train_loader:
                inputs = batch['inputs'].to(device)
                labels = batch['labels'].to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item()
            scheduler.step()

            part_probs, part_labels_fold, _ = evaluate_fold(model, val_loader, device)
            if len(np.unique(part_labels_fold)) > 1:
                auc = roc_auc_score(part_labels_fold, part_probs)
                fpr, tpr, thresholds = roc_curve(part_labels_fold, part_probs)
                opt_idx = np.argmax(tpr - fpr)
                opt_thresh = thresholds[opt_idx]
                preds_opt = (part_probs >= opt_thresh).astype(int)
                f1_opt = f1_score(part_labels_fold, preds_opt, zero_division=0)
            else:
                auc, f1_opt = 0.5, 0.0

            score = 0.4 * auc + 0.6 * f1_opt
            if score > best_score + 0.01:
                best_score = score
                best_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
                marker = '<-- best'
            else:
                patience_counter += 1
                marker = ''

            avg_loss = total_loss / len(train_loader)
            print(f'  Epoch {epoch+1:02d} | loss {avg_loss:.4f} | AUC {auc:.3f} | F1opt {f1_opt:.3f} {marker}')

            if patience_counter >= PATIENCE:
                print(f'  Early stopping at epoch {epoch+1}')
                break

        model.load_state_dict(best_state)
        part_probs, part_labels_fold, val_pids = evaluate_fold(model, val_loader, device)

        torch.save(model.state_dict(), str(RESULTS_DIR / f'ast_pd_v3_fold{fold+1}.pt'))

        for i, pid in enumerate(val_pids):
            idx_oof = np.where(unique_participants == pid)[0][0]
            all_oof_probs[idx_oof] = part_probs[i]

        fold_auc = roc_auc_score(part_labels_fold, part_probs)
        fpr, tpr, thresholds = roc_curve(part_labels_fold, part_probs)
        opt_idx = np.argmax(tpr - fpr)
        opt_thresh = thresholds[opt_idx]
        preds_opt = (part_probs >= opt_thresh).astype(int)

        fold_results.append({
            'fold': fold + 1,
            'auc': float(fold_auc),
            'f1_opt': float(f1_score(part_labels_fold, preds_opt, zero_division=0)),
            'recall_opt': float(recall_score(part_labels_fold, preds_opt, zero_division=0)),
            'precision_opt': float(precision_score(part_labels_fold, preds_opt, zero_division=0)),
            'threshold': float(opt_thresh)
        })

        print(f'Fold {fold+1}: AUC={fold_auc:.4f}, F1={fold_results[-1]["f1_opt"]:.4f}')

        del model, optimizer, train_ds, val_ds
        torch.cuda.empty_cache()

    total_time = time.time() - total_start
    print(f'\nTotal CV time: {total_time/60:.1f} minutes')


In [ ]:
#8 - CV Summary and OOF evaluation
from scipy import stats
from sklearn.metrics import confusion_matrix, accuracy_score

aucs = [r['auc'] for r in fold_results]
f1s = [r['f1_opt'] for r in fold_results]
recalls = [r['recall_opt'] for r in fold_results]
precisions = [r['precision_opt'] for r in fold_results]

n = N_FOLDS
t_crit = stats.t.ppf(0.975, df=n - 1)

print('Per-fold results:')
for r in fold_results:
    print(f'  Fold {r["fold"]}: AUC={r["auc"]:.4f}  F1={r["f1_opt"]:.4f}  '
          f'Rec={r["recall_opt"]:.4f}  Prec={r["precision_opt"]:.4f}  Thr={r["threshold"]:.3f}')

print('\nMean +/- SD [95% CI]:')
for name, arr in [('AUC', aucs), ('F1', f1s), ('Recall', recalls), ('Precision', precisions)]:
    m = np.mean(arr)
    sd = np.std(arr, ddof=1)
    se = sd / np.sqrt(n)
    ci_lo = m - t_crit * se
    ci_hi = m + t_crit * se
    print(f'  {name}: {m:.4f} ({sd:.4f}) [{ci_lo:.4f}, {ci_hi:.4f}]')

# OOF metrics
oof_auc = roc_auc_score(all_oof_labels, all_oof_probs)
fpr, tpr, thresholds = roc_curve(all_oof_labels, all_oof_probs)
opt_idx = np.argmax(tpr - fpr)
oof_thresh = thresholds[opt_idx]
oof_preds = (all_oof_probs >= oof_thresh).astype(int)

oof_f1 = f1_score(all_oof_labels, oof_preds, zero_division=0)
oof_rec = recall_score(all_oof_labels, oof_preds, zero_division=0)
oof_prec = precision_score(all_oof_labels, oof_preds, zero_division=0)
oof_acc = accuracy_score(all_oof_labels, oof_preds)
cm = confusion_matrix(all_oof_labels, oof_preds)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

print(f'\nOOF AUC:         {oof_auc:.4f}')
print(f'OOF F1:          {oof_f1:.4f}')
print(f'OOF Recall:      {oof_rec:.4f}')
print(f'OOF Precision:   {oof_prec:.4f}')
print(f'OOF Specificity: {specificity:.4f}')
print(f'OOF Accuracy:    {oof_acc:.4f}')
print(f'Threshold:       {oof_thresh:.3f}')
print(f'\nConfusion Matrix: TP={tp} FP={fp} FN={fn} TN={tn}')

# Save results
np.savez(str(RESULTS_DIR / 'ast_pd_v3_cv_results.npz'),
    oof_probs=all_oof_probs,
    oof_labels=all_oof_labels,
    participant_ids=unique_participants,
    fold_aucs=np.array(aucs),
    fold_f1s=np.array(f1s),
    fold_recalls=np.array(recalls),
    fold_precisions=np.array(precisions),
    fold_thresholds=np.array([r['threshold'] for r in fold_results]),
    oof_auc=np.array(oof_auc),
    oof_thresh=np.array(oof_thresh),
    norm_means=np.array([norm_stats[f+1]['mean'] for f in range(N_FOLDS)]),
    norm_stds=np.array([norm_stats[f+1]['std'] for f in range(N_FOLDS)]),
)
print(f'\nSaved: {RESULTS_DIR}/ast_pd_v3_cv_results.npz')
print(f'Saved: {RESULTS_DIR}/ast_pd_v3_fold{{1-5}}.pt')

In [ ]:
#9 - Metadata evaluation (age + sex logistic regression + late fusion)
from sklearn.linear_model import LogisticRegression

demo = pd.read_csv(DEMO_PATH, sep='\t')
demo['participant_id'] = demo['participant_id'].astype(str).str.zfill(6)
# Coerce age to numeric and deduplicate. We deliberately do NOT replace '90 and above' with 90
# here; doing so would include a single 90+ year-old control participant whose inclusion produces
# numerically different DeLong/propensity results than the metadata_experiment_v3_fixed.npz
# baseline used in the manuscript. Coercing such non-numeric values to NaN reproduces the paper.
demo['age'] = pd.to_numeric(demo['age'], errors='coerce')
demo = demo.sort_values('age', na_position='last').groupby('participant_id').first().reset_index()

# Build metadata features for our cohort
meta_df = pd.DataFrame({'participant_id': unique_participants, 'label': participant_labels})
meta_df = meta_df.merge(demo[['participant_id', 'age', 'sex_at_birth']], on='participant_id', how='left')
meta_df['sex_numeric'] = (meta_df['sex_at_birth'] == 'Male').astype(int)
meta_df = meta_df.dropna(subset=['age', 'sex_numeric'])

print(f'Participants with metadata: {len(meta_df)}')

# 5-fold CV for metadata-only model (same splits)
meta_participants = meta_df['participant_id'].values
meta_labels = meta_df['label'].values
meta_features = meta_df[['age', 'sex_numeric']].values

meta_oof_probs = np.zeros(len(meta_df), dtype=np.float32)

skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf_meta.split(meta_participants, meta_labels)):
    lr = LogisticRegression(class_weight='balanced', max_iter=1000)
    lr.fit(meta_features[train_idx], meta_labels[train_idx])
    meta_oof_probs[val_idx] = lr.predict_proba(meta_features[val_idx])[:, 1]

meta_auc = roc_auc_score(meta_labels, meta_oof_probs)
fpr_m, tpr_m, thr_m = roc_curve(meta_labels, meta_oof_probs)
opt_idx_m = np.argmax(tpr_m - fpr_m)
meta_preds = (meta_oof_probs >= thr_m[opt_idx_m]).astype(int)
meta_f1 = f1_score(meta_labels, meta_preds, zero_division=0)

print(f'\nMetadata-only AUC: {meta_auc:.4f}')
print(f'Metadata-only F1:  {meta_f1:.4f}')

# Late fusion: combine AST probs + metadata probs
# Match participants between AST and metadata
ast_prob_map = dict(zip(unique_participants, all_oof_probs))
fused_probs = []
fused_labels = []
for i, pid in enumerate(meta_participants):
    if pid in ast_prob_map:
        fused_probs.append(0.5 * ast_prob_map[pid] + 0.5 * meta_oof_probs[i])
        fused_labels.append(meta_labels[i])

fused_probs = np.array(fused_probs)
fused_labels = np.array(fused_labels)
fused_auc = roc_auc_score(fused_labels, fused_probs)
fpr_f, tpr_f, thr_f = roc_curve(fused_labels, fused_probs)
opt_idx_f = np.argmax(tpr_f - fpr_f)
fused_preds = (fused_probs >= thr_f[opt_idx_f]).astype(int)
fused_f1 = f1_score(fused_labels, fused_preds, zero_division=0)

print(f'\nAST+Metadata AUC:  {fused_auc:.4f}')
print(f'AST+Metadata F1:   {fused_f1:.4f}')

print(f'\nSummary:')
print(f'  AST only:     AUC={oof_auc:.4f}  F1={oof_f1:.4f}')
print(f'  Metadata only: AUC={meta_auc:.4f}  F1={meta_f1:.4f}')
print(f'  AST+Metadata: AUC={fused_auc:.4f}  F1={fused_f1:.4f}')

In [ ]:
#10 - Per-task contribution analysis
# Re-run inference on each task separately using fold models
import json

task_results = {}

for task in SELECTED_TASKS:
    task_mask = tasks_raw == task
    if task_mask.sum() == 0:
        continue
    X_task = X_raw[task_mask]
    y_task = y_raw[task_mask]
    p_task = participants_raw[task_mask]

    # Recording-level AUC from OOF: match each recording's participant to their OOF prob
    rec_probs = []
    rec_labels = []
    for i, pid in enumerate(p_task):
        idx_p = np.where(unique_participants == pid)[0]
        if len(idx_p) > 0:
            rec_probs.append(all_oof_probs[idx_p[0]])
            rec_labels.append(y_task[i])

    if len(np.unique(rec_labels)) > 1:
        task_auc = roc_auc_score(rec_labels, rec_probs)
    else:
        task_auc = float('nan')

    task_results[task] = {
        'recordings': int(task_mask.sum()),
        'participants': len(np.unique(p_task)),
        'auc': float(task_auc)
    }
    print(f'{task}: {task_mask.sum()} recs, {len(np.unique(p_task))} ppl, AUC={task_auc:.4f}')

# Save task results
with open(str(RESULTS_DIR / 'ast_pd_v3_per_task.json'), 'w') as f:
    json.dump(task_results, f, indent=2)
print(f'\nSaved: {RESULTS_DIR}/ast_pd_v3_per_task.json')

## Reviewer-response analyses (added May 2026)

The cells below add three robustness analyses requested by reviewers and reference
the saved fold checkpoints / OOF predictions written by cell #7. They are
re-runnable in isolation: `papermill` will load checkpoints from `results/v3/` and
skip retraining (see idempotency wrapper at the top of cell #7).

- **#11** Threshold-leakage sensitivity (LOFO Youden + fixed=0.5)
- **#12** Training-partition Youden's J (loads fold checkpoints, runs inference on training partition)
- **#13** DeLong's test for paired AUC (AST vs age-only vs metadata)
- **#14** Propensity-score matching (USA-only ages 60-80; degrades gracefully if infeasible)


In [ ]:
#11 - Threshold-leakage sensitivity: LOFO Youden + fixed=0.5 (reviewer #7 response)
# Two leak-free alternatives to the original Youden-on-aggregated-OOF threshold:
#   (1) LOFO Youden's J: threshold for fold k is computed from the OOF predictions of the OTHER 4 folds.
#   (2) Fixed = 0.5: no optimization, no leakage at all.
# AUC-ROC is threshold-independent and unchanged across strategies.
import json as _json
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, confusion_matrix, accuracy_score

def _youden(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr - fpr)])

def _metrics(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return dict(
        f1=float(f1_score(y, pred, zero_division=0)),
        precision=float(precision_score(y, pred, zero_division=0)),
        recall=float(recall_score(y, pred, zero_division=0)),
        specificity=float(tn/(tn+fp)) if (tn+fp)>0 else 0.0,
        accuracy=float(accuracy_score(y, pred)),
        tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn),
    )

# Reproduce the same fold assignments used during training
_skf_t = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
_fold_for_pid = {}
for _f, (_, _vidx) in enumerate(_skf_t.split(unique_participants, participant_labels)):
    for _pid in unique_participants[_vidx]:
        _fold_for_pid[_pid] = _f
_fold_idx = np.array([_fold_for_pid[p] for p in unique_participants])

# Strategy 1: LOFO Youden's J
_lofo_T = []
for k in range(N_FOLDS):
    _other = (_fold_idx != k)
    _lofo_T.append(_youden(all_oof_labels[_other], all_oof_probs[_other]))

_lofo_preds = np.zeros(len(unique_participants), dtype=int)
for _i in range(len(unique_participants)):
    _lofo_preds[_i] = int(all_oof_probs[_i] >= _lofo_T[_fold_idx[_i]])

# Strategy 2: Fixed = 0.5
_fixed_preds = (all_oof_probs >= 0.5).astype(int)

# Original (leaky) — Youden on aggregated OOF
_old_T = _youden(all_oof_labels, all_oof_probs)
_old_preds = (all_oof_probs >= _old_T).astype(int)

threshold_strategies = {
    'condition': 'pd',
    'oof_auc': float(roc_auc_score(all_oof_labels, all_oof_probs)),
    'n_participants': int(len(unique_participants)),
    'old_leaky_OOF': dict(threshold=float(_old_T), **_metrics(all_oof_labels, _old_preds)),
    'lofo': dict(thresholds=[float(t) for t in _lofo_T], mean_T=float(np.mean(_lofo_T)), sd_T=float(np.std(_lofo_T, ddof=1)), **_metrics(all_oof_labels, _lofo_preds)),
    'fixed_05': dict(threshold=0.5, **_metrics(all_oof_labels, _fixed_preds)),
}

print(f'AUC (threshold-independent): {threshold_strategies["oof_auc"]:.4f}  N={len(unique_participants)}')
print(f'  LEAKY (Youden on aggregated OOF):  T={_old_T:.4f}  F1={threshold_strategies["old_leaky_OOF"]["f1"]:.4f}  P={threshold_strategies["old_leaky_OOF"]["precision"]:.4f}  R={threshold_strategies["old_leaky_OOF"]["recall"]:.4f}')
print(f'  LOFO  (Youden on other folds):     T={np.mean(_lofo_T):.4f} (SD {np.std(_lofo_T, ddof=1):.4f})  F1={threshold_strategies["lofo"]["f1"]:.4f}  P={threshold_strategies["lofo"]["precision"]:.4f}  R={threshold_strategies["lofo"]["recall"]:.4f}')
print(f'  FIXED 0.5:                         F1={threshold_strategies["fixed_05"]["f1"]:.4f}  P={threshold_strategies["fixed_05"]["precision"]:.4f}  R={threshold_strategies["fixed_05"]["recall"]:.4f}')

with open(str(RESULTS_DIR / 'lofo_and_fixed_threshold_metrics_pd.json'), 'w') as f:
    _json.dump(threshold_strategies, f, indent=2)
print(f'Saved: {RESULTS_DIR}/lofo_and_fixed_threshold_metrics_pd.json')


In [ ]:
#12 - Training-partition Youden's J (per fold, no leakage; loads fold checkpoints)
# For each fold k: load fold_k checkpoint, run inference on the TRAINING participants,
# compute Youden's J on those training-set participant-level probs to get T_k.
# Apply T_k to fold-k validation OOF predictions; aggregate across folds.
import json as _json

@torch.no_grad()
def _fold_inference(model, X, batch_size=16):
    model.eval()
    probs_all = []
    for i in range(0, len(X), batch_size):
        xb = torch.tensor(X[i:i+batch_size], dtype=torch.float32, device=device)
        logits = model(xb)
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        probs_all.append(probs)
    return np.concatenate(probs_all)

def _participant_aggregate(probs, labels, parts):
    df = pd.DataFrame({'pid': parts, 'prob': probs, 'label': labels})
    g = df.groupby('pid').agg({'prob': 'mean', 'label': 'first'}).reset_index()
    return g['pid'].values, g['prob'].values.astype(np.float32), g['label'].values.astype(int)

_skf_t2 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_train_thresh = []
per_fold_records = []

_pid_to_oof_idx = {p: i for i, p in enumerate(unique_participants)}

for _fold, (_train_idx, _val_idx) in enumerate(_skf_t2.split(unique_participants, participant_labels)):
    _train_parts = unique_participants[_train_idx]
    _val_parts = unique_participants[_val_idx]
    _train_mask = np.isin(participants_raw, _train_parts)

    _X_train = X_raw[_train_mask]
    _y_train = y_raw[_train_mask]
    _parts_train = participants_raw[_train_mask]

    _X_train_ast = np.stack([resize_spectrogram(x) for x in _X_train])
    _fmean = norm_stats[_fold + 1]['mean']
    _fstd = norm_stats[_fold + 1]['std']
    _X_train_ast = (_X_train_ast - _fmean) / (_fstd + 1e-8)

    _ckpt = RESULTS_DIR / f'ast_pd_v3_fold{_fold+1}.pt'
    print(f'Fold {_fold+1}: loading {_ckpt.name}, mean={_fmean:.4f}, std={_fstd:.4f}')
    _model = ASTClassifier(num_classes=2, pretrained=True, freeze_base=False).to(device)
    _sd = torch.load(_ckpt, map_location=device, weights_only=False)
    if isinstance(_sd, dict) and 'state_dict' in _sd:
        _sd = _sd['state_dict']
    _model.load_state_dict(_sd, strict=True)

    _probs_train = _fold_inference(_model, _X_train_ast, batch_size=16)
    del _model
    torch.cuda.empty_cache()

    _, _train_part_probs, _train_part_labels = _participant_aggregate(_probs_train, _y_train, _parts_train)
    _T_k = _youden(_train_part_labels, _train_part_probs)
    fold_train_thresh.append(_T_k)

    _val_probs = np.array([all_oof_probs[_pid_to_oof_idx[p]] for p in _val_parts])
    _val_labels = np.array([all_oof_labels[_pid_to_oof_idx[p]] for p in _val_parts])
    _val_preds = (_val_probs >= _T_k).astype(int)

    per_fold_records.append({
        'fold': _fold + 1,
        'T_train': float(_T_k),
        'n_train_parts': int(len(_train_part_probs)),
        'n_val_parts': int(len(_val_parts)),
        'val_f1': float(f1_score(_val_labels, _val_preds, zero_division=0)),
        'val_recall': float(recall_score(_val_labels, _val_preds, zero_division=0)),
        'val_precision': float(precision_score(_val_labels, _val_preds, zero_division=0)),
    })
    print(f'  T_train={_T_k:.4f}  fold val F1={per_fold_records[-1]["val_f1"]:.4f}')

# Aggregate using fold-specific T_k
_pid_to_fold = {}
for _fold, (_, _val_idx) in enumerate(_skf_t2.split(unique_participants, participant_labels)):
    for _pid in unique_participants[_val_idx]:
        _pid_to_fold[_pid] = _fold

_new_preds = np.array([
    int(all_oof_probs[i] >= fold_train_thresh[_pid_to_fold[unique_participants[i]]])
    for i in range(len(unique_participants))
], dtype=int)
_train_thresh_metrics = _metrics(all_oof_labels, _new_preds)

training_threshold_summary = {
    'condition': 'pd',
    'oof_auc_threshold_independent': float(threshold_strategies['oof_auc']),
    'old_threshold_aggregated_OOF': float(threshold_strategies['old_leaky_OOF']['threshold']),
    'old_F1': float(threshold_strategies['old_leaky_OOF']['f1']),
    'old_precision': float(threshold_strategies['old_leaky_OOF']['precision']),
    'old_recall': float(threshold_strategies['old_leaky_OOF']['recall']),
    'fold_train_thresholds': [float(t) for t in fold_train_thresh],
    'mean_train_threshold': float(np.mean(fold_train_thresh)),
    'sd_train_threshold': float(np.std(fold_train_thresh, ddof=1)),
    'new_F1_train_threshold': float(_train_thresh_metrics['f1']),
    'new_precision_train_threshold': float(_train_thresh_metrics['precision']),
    'new_recall_train_threshold': float(_train_thresh_metrics['recall']),
    'new_specificity_train_threshold': float(_train_thresh_metrics['specificity']),
    'new_accuracy_train_threshold': float(_train_thresh_metrics['accuracy']),
    'confusion_matrix': {'tp': _train_thresh_metrics['tp'], 'fp': _train_thresh_metrics['fp'],
                         'fn': _train_thresh_metrics['fn'], 'tn': _train_thresh_metrics['tn']},
    'per_fold_records': per_fold_records,
}
with open(str(RESULTS_DIR / 'training_threshold_metrics_pd.json'), 'w') as f:
    _json.dump(training_threshold_summary, f, indent=2)
print(f'\nMean train threshold: {np.mean(fold_train_thresh):.4f} (SD {np.std(fold_train_thresh, ddof=1):.4f})')
print(f'New F1: {_train_thresh_metrics["f1"]:.4f}  P: {_train_thresh_metrics["precision"]:.4f}  R: {_train_thresh_metrics["recall"]:.4f}')
print(f'Saved: {RESULTS_DIR}/training_threshold_metrics_pd.json')


In [ ]:
#13 - DeLong's test for paired AUC comparisons (Sun & Xu 2014 fast O(N log N))
# Compare AST vs age-only vs metadata (age+sex) under the same participants and folds.
# Reported in Results to support the "age alone is statistically indistinguishable from AST" claim.
from scipy import stats as _stats

def _compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N_ = len(x)
    T = np.zeros(N_); i = 0
    while i < N_:
        j = i
        while j < N_ and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N_); T2[J] = T
    return T2

def _fast_delong(preds, m):
    n = preds.shape[1] - m
    pos = preds[:, :m]; neg = preds[:, m:]
    k = preds.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r] = _compute_midrank(pos[r])
        ty[r] = _compute_midrank(neg[r])
        tz[r] = _compute_midrank(preds[r])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    return aucs, sx / m + sy / n

def delong_test(labels, preds_a, preds_b):
    order = (-labels).argsort()
    m_ = int(labels.sum())
    preds = np.vstack([preds_a[order], preds_b[order]])
    aucs, cov = _fast_delong(preds, m_)
    diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    se = np.sqrt(max(var, 0.0))
    z = diff / se if se > 0 else (np.inf if diff > 0 else -np.inf if diff < 0 else 0.0)
    p = 2 * (1 - _stats.norm.cdf(abs(z)))
    return float(aucs[0]), float(aucs[1]), float(z), float(p), float(diff - 1.96 * se), float(diff + 1.96 * se)

# Aligned arrays: use the metadata cohort (subset of AST cohort with valid age + sex)
_ast_prob_map_full = dict(zip(unique_participants, all_oof_probs))
_aligned_pids = meta_participants
_aligned_labels = meta_labels.astype(int)
_aligned_ast = np.array([_ast_prob_map_full[p] for p in _aligned_pids])
_aligned_meta = meta_oof_probs

# Recompute age-only OOF using the same 5-fold protocol used for metadata
_age = meta_features[:, 0:1]
_age_oof = np.zeros(len(_aligned_pids), dtype=float)
_skf_age = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for _f, (_tr, _vl) in enumerate(_skf_age.split(_age, _aligned_labels)):
    _lr = LogisticRegression(class_weight='balanced', max_iter=1000)
    _lr.fit(_age[_tr], _aligned_labels[_tr])
    _age_oof[_vl] = _lr.predict_proba(_age[_vl])[:, 1]

print(f'AST AUC (aligned): {roc_auc_score(_aligned_labels, _aligned_ast):.4f}')
print(f'Age-only AUC: {roc_auc_score(_aligned_labels, _age_oof):.4f}')
print(f'Metadata AUC: {roc_auc_score(_aligned_labels, _aligned_meta):.4f}')

delong_results = {}
print('\n--- Full cohort ---')
for _name, _a, _b in [
    ('AST_full vs AgeOnly_full', _aligned_ast, _age_oof),
    ('AST_full vs Metadata_full', _aligned_ast, _aligned_meta),
    ('AgeOnly_full vs Metadata_full', _age_oof, _aligned_meta),
]:
    auc_a, auc_b, z, p, ci_lo, ci_hi = delong_test(_aligned_labels, _a, _b)
    delong_results[_name] = dict(auc_a=auc_a, auc_b=auc_b, diff=auc_a - auc_b, z=z, p=p,
                                 ci_lo=ci_lo, ci_hi=ci_hi, n=int(len(_aligned_labels)))
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  {_name}: {auc_a:.4f} vs {auc_b:.4f}  diff={auc_a-auc_b:+.4f} [{ci_lo:+.4f}, {ci_hi:+.4f}]  p={p:.4g} {sig}')

# Age-restricted (60-80) subgroup
_ages_arr = meta_features[:, 0]
_sub_60_80 = (_ages_arr >= 60) & (_ages_arr <= 80)
print(f'\n--- Age-restricted (60-80) N: {_sub_60_80.sum()} ---')
for _name, _a, _b in [
    ('AST_age60-80 vs AgeOnly_age60-80', _aligned_ast[_sub_60_80], _age_oof[_sub_60_80]),
    ('AST_age60-80 vs Metadata_age60-80', _aligned_ast[_sub_60_80], _aligned_meta[_sub_60_80]),
]:
    if len(np.unique(_aligned_labels[_sub_60_80])) < 2:
        print(f'  {_name}: skipped (single class in subgroup)')
        continue
    auc_a, auc_b, z, p, ci_lo, ci_hi = delong_test(_aligned_labels[_sub_60_80], _a, _b)
    delong_results[_name] = dict(auc_a=auc_a, auc_b=auc_b, diff=auc_a - auc_b, z=z, p=p,
                                 ci_lo=ci_lo, ci_hi=ci_hi, n=int(_sub_60_80.sum()))
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  {_name}: {auc_a:.4f} vs {auc_b:.4f}  diff={auc_a-auc_b:+.4f} [{ci_lo:+.4f}, {ci_hi:+.4f}]  p={p:.4g} {sig}')


In [ ]:
#14 - Propensity-score matching (USA-only ages 60-80; reviewer #8 strong-form response)
# 1:1 nearest-neighbor matching on logistic-regression propensity (age + sex), no replacement,
# caliper = 0.2 * SD of propensity. Degrades gracefully when subgroup lacks cases.
import json as _json

_demo_full = pd.read_csv(DEMO_PATH, sep='\t')
_demo_full['participant_id'] = _demo_full['participant_id'].astype(str).str.zfill(6)
_demo_full['age_num'] = pd.to_numeric(_demo_full['age'].replace({'90 and above': 90}), errors='coerce')
_demo_agg = _demo_full.sort_values('age_num', na_position='last').groupby('participant_id').first().reset_index()

_match_df = pd.DataFrame({
    'pid': _aligned_pids, 'label': _aligned_labels,
    'ast': _aligned_ast, 'age_only': _age_oof, 'meta': _aligned_meta,
    'age': _ages_arr, 'sex': meta_features[:, 1].astype(int),
}).merge(_demo_agg[['participant_id', 'country']], left_on='pid', right_on='participant_id', how='left')

_country_str = _match_df['country'].astype(str)
_usa_mask = _country_str.str.upper().str.startswith('US') | (_country_str.str.lower() == 'united states')
_sub = _match_df[(_match_df['age'] >= 60) & (_match_df['age'] <= 80) & _usa_mask].reset_index(drop=True)
print(f'USA-only age 60-80 subgroup: N={len(_sub)}, cases={int(_sub.label.sum())}, controls={int((_sub.label==0).sum())}')

propensity_results = None
if int(_sub.label.sum()) >= 5 and int((_sub.label == 0).sum()) >= 5:
    _X_p = _sub[['age', 'sex']].values
    _y_p = _sub['label'].values
    _lr_p = LogisticRegression(class_weight='balanced', max_iter=1000)
    _lr_p.fit(_X_p, _y_p)
    _sub['propensity'] = _lr_p.predict_proba(_X_p)[:, 1]

    _smd_pre = (_sub.loc[_sub.label==1, 'age'].mean() - _sub.loc[_sub.label==0, 'age'].mean()) / np.sqrt(0.5*(_sub.loc[_sub.label==1, 'age'].var() + _sub.loc[_sub.label==0, 'age'].var()))
    print(f'Pre-match SMD age: {_smd_pre:.3f}')

    _cases = _sub[_sub.label==1].reset_index(drop=True)
    _ctrls = _sub[_sub.label==0].reset_index(drop=True)
    _caliper = 0.2 * _sub['propensity'].std()
    print(f'Caliper (0.2 * SD propensity): {_caliper:.4f}')

    _used = set()
    _matches = []
    _rng = np.random.default_rng(42)
    _case_order = _rng.permutation(len(_cases))
    for _ci in _case_order:
        _case = _cases.iloc[_ci]
        _cand = [j for j in range(len(_ctrls)) if j not in _used]
        if not _cand:
            break
        _dists = np.array([abs(_case['propensity'] - _ctrls.iloc[j]['propensity']) for j in _cand])
        _best = int(np.argmin(_dists))
        if _dists[_best] > _caliper:
            continue
        _used.add(_cand[_best])
        _matches.append((_case, _ctrls.iloc[_cand[_best]]))

    _n_pairs = len(_matches)
    print(f'Matched pairs: {_n_pairs}')

    if _n_pairs >= 10:
        _matched_df = pd.DataFrame([m[0] for m in _matches] + [m[1] for m in _matches])
        _smd_post = (_matched_df.loc[_matched_df.label==1, 'age'].mean() - _matched_df.loc[_matched_df.label==0, 'age'].mean()) / np.sqrt(0.5*(_matched_df.loc[_matched_df.label==1, 'age'].var() + _matched_df.loc[_matched_df.label==0, 'age'].var()))
        print(f'Post-match SMD age: {_smd_post:.3f}')

        _y_m = _matched_df['label'].values
        _matched_aucs = {
            'AST': float(roc_auc_score(_y_m, _matched_df['ast'])),
            'Age_only': float(roc_auc_score(_y_m, _matched_df['age_only'])),
            'Metadata_age_sex': float(roc_auc_score(_y_m, _matched_df['meta'])),
        }
        for k, v in _matched_aucs.items():
            print(f'  {k}: {v:.4f}')

        auc_a, auc_b, z, p, ci_lo, ci_hi = delong_test(_y_m, _matched_df['ast'].values, _matched_df['age_only'].values)
        print(f'DeLong AST vs Age-only (matched): diff={auc_a-auc_b:+.4f} [{ci_lo:+.4f}, {ci_hi:+.4f}]  p={p:.4g}')

        propensity_results = {
            'usa_age_60_80_n_total': int(len(_sub)),
            'usa_age_60_80_n_pd': int(_sub.label.sum()),
            'usa_age_60_80_n_ctrl': int((_sub.label==0).sum()),
            'mean_age_pd_pre': float(_sub.loc[_sub.label==1, 'age'].mean()),
            'mean_age_ctrl_pre': float(_sub.loc[_sub.label==0, 'age'].mean()),
            'smd_age_pre': float(_smd_pre),
            'caliper': float(_caliper),
            'n_matched_pairs': int(_n_pairs),
            'n_matched_total': int(len(_matched_df)),
            'mean_age_pd_post': float(_matched_df.loc[_matched_df.label==1, 'age'].mean()),
            'mean_age_ctrl_post': float(_matched_df.loc[_matched_df.label==0, 'age'].mean()),
            'smd_age_post': float(_smd_post),
            'matched_aucs': _matched_aucs,
            'delong_ast_vs_age_matched': {
                'auc_ast': float(auc_a), 'auc_age': float(auc_b),
                'diff': float(auc_a - auc_b), 'ci_lo': float(ci_lo), 'ci_hi': float(ci_hi), 'p': float(p)
            }
        }
    else:
        print(f'Too few matches ({_n_pairs}); cannot reliably compute matched AUCs')
        propensity_results = {'reason': f'only {_n_pairs} matched pairs', 'n_total': int(len(_sub)),
                               'n_cases': int(_sub.label.sum()), 'n_ctrls': int((_sub.label==0).sum())}
else:
    print(f'Insufficient cases or controls in USA-only 60-80 subgroup; skipping matching.')
    propensity_results = {'reason': 'insufficient cases or controls', 'n_total': int(len(_sub)),
                           'n_cases': int(_sub.label.sum()), 'n_ctrls': int((_sub.label==0).sum())}

_combined = {'condition': 'pd', 'delong': delong_results, 'propensity': propensity_results}
with open(str(RESULTS_DIR / 'delong_and_propensity_pd.json'), 'w') as f:
    _json.dump(_combined, f, indent=2, default=float)
print(f'Saved: {RESULTS_DIR}/delong_and_propensity_pd.json')
